In [1]:
from pyspark.sql import SparkSession

PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Permormance_Analytics") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [2]:
from pyspark.sql.functions import *

spark.sql("""
SELECT 
    COUNT(*) as total_files,
    ROUND(AVG(file_size_in_bytes) / 1024, 2) as avg_size_kb,
    SUM(record_count) as total_records
FROM my_catalog.default.user_events_file_based.files
""").show()

spark.sql("""
SELECT 
    file_path, 
    file_size_in_bytes, 
    record_count 
FROM my_catalog.default.user_events_file_based.files 
LIMIT 5
""").show(truncate=False)

+-----------+-----------+-------------+
|total_files|avg_size_kb|total_records|
+-----------+-----------+-------------+
|         13|        7.5|         2437|
+-----------+-----------+-------------+

+-----------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------+
|file_path                                                                                                                                      |file_size_in_bytes|record_count|
+-----------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------+
|s3a://warehouse/default/user_events_file_based/data/timestamp_hour=2026-05-23-19/00078-46369-78855a04-b293-4d53-90e6-521335d7b348-00001.parquet|9253              |247         |
|s3a://warehouse/default/user_events_file_based/data/timestamp_hour=2026-05-23-19/00078

In [2]:
import time
import os 
from datetime import datetime

LOG_FILE = "benchmark_results_file_based.csv"

if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        f.write("timestamp,files_count,execution_time_agg,execution_time_single\n")
    print(f"log file created: {LOG_FILE}")

def benchmark_read(table_name="my_catalog.default.user_events_file_based"):
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"run ts: {current_time}")

    
    start_time_agg = time.time()
    spark.sql(f"""
        SELECT 
            shop,
            device_type,
            event_type,
            COUNT(DISTINCT session_id) as unique_sessions, 
            COUNT(DISTINCT user_id) as unique_users,
            SUM(price) as total_revenue, 
            AVG(price) as avg_check,
            MIN(timestamp) as first_interaction,
            MAX(timestamp) as latest_interaction
        FROM {table_name}
        WHERE shop IN ('rozetka', 'foxtrot', 'citrus') AND device_type IN ('ios', 'android')
        GROUP BY 1,2,3
        ORDER BY 
            total_revenue DESC,
            unique_sessions DESC
    """).collect() # action for spark to actually calculate things
    
    end_time_agg = time.time()
    execution_time_agg = end_time_agg - start_time_agg
    files_count = spark.sql("SELECT COUNT(*) FROM my_catalog.default.user_events_file_based.files").collect()[0][0]

    start_time_single = time.time()
    spark.sql(f"""
        SELECT * FROM {table_name}
        WHERE shop = 'rozetka' 
          AND device_type = 'mobile_web' 
          AND event_type = 'purchase'
    """).collect()
    end_time_single = time.time()

    execution_time_single = end_time_single - start_time_single

    with open(LOG_FILE, "a") as f:
        f.write(f"{current_time},{files_count},{execution_time_agg:.4f},{execution_time_single:.4f}\n")
    
    print(f"FILES NUMBER: {files_count} /// TIME EXECUTING (AGG): {execution_time_agg:.4f}")
    print(f"FILES NUMBER: {files_count} /// TIME EXECUTING (SINGLE): {execution_time_single:.4f}")


In [ ]:

INTERVAL_MINUTES = 1

try:
    while True:
        benchmark_read()
        print(f"waiting set interval")
        time.sleep(INTERVAL_MINUTES * 60)
except KeyboardInterrupt:
    print("interrupted")

run ts: 2026-05-23 22:13:36
FILES NUMBER: 3 /// TIME EXECUTING (AGG): 0.5214
FILES NUMBER: 3 /// TIME EXECUTING (SINGLE): 0.1370
waiting set interval
run ts: 2026-05-23 22:14:37
FILES NUMBER: 15 /// TIME EXECUTING (AGG): 0.4298
FILES NUMBER: 15 /// TIME EXECUTING (SINGLE): 0.1660
waiting set interval
run ts: 2026-05-23 22:15:38
FILES NUMBER: 27 /// TIME EXECUTING (AGG): 0.6176
FILES NUMBER: 27 /// TIME EXECUTING (SINGLE): 0.2125
waiting set interval
run ts: 2026-05-23 22:16:39
FILES NUMBER: 39 /// TIME EXECUTING (AGG): 0.6532
FILES NUMBER: 39 /// TIME EXECUTING (SINGLE): 0.2940
waiting set interval
run ts: 2026-05-23 22:17:40
FILES NUMBER: 51 /// TIME EXECUTING (AGG): 1.0572
FILES NUMBER: 51 /// TIME EXECUTING (SINGLE): 0.3457
waiting set interval
run ts: 2026-05-23 22:18:41
FILES NUMBER: 64 /// TIME EXECUTING (AGG): 1.0192
FILES NUMBER: 64 /// TIME EXECUTING (SINGLE): 0.5173
waiting set interval
run ts: 2026-05-23 22:19:43
FILES NUMBER: 76 /// TIME EXECUTING (AGG): 0.8437
FILES NUMBER

In [3]:
TABLE_NAME = "my_catalog.default.user_events_file_based"

stats_df = spark.sql(f"""
    SELECT 
        COUNT(*) as current_files_count,
        MIN(file_size_in_bytes) / 1048576.0 AS min_size_mb,
        AVG(file_size_in_bytes) / 1048576.0 AS mean_size_mb,
        percentile_approx(file_size_in_bytes, 0.25) / 1048576.0 AS q1_size_mb,
        percentile_approx(file_size_in_bytes, 0.50) / 1048576.0 AS median_size_mb,
        percentile_approx(file_size_in_bytes, 0.75) / 1048576.0 AS q3_size_mb,
        MAX(file_size_in_bytes) / 1048576.0 AS max_size_mb
    FROM {TABLE_NAME}.files
""")

stats_df.show()

+-------------------+-----------+--------------------+-----------+--------------+-----------+-----------+
|current_files_count|min_size_mb|        mean_size_mb| q1_size_mb|median_size_mb| q3_size_mb|max_size_mb|
+-------------------+-----------+--------------------+-----------+--------------+-----------+-----------+
|                 15|0.002424240|0.004834747314453125|0.005058289|   0.005137444|0.005160332|0.005182266|
+-------------------+-----------+--------------------+-----------+--------------+-----------+-----------+

